# Claude Agent SDK: connecting to MCP servers

**MCP (Model Context Protocol)** is an open standard for plugging tools and data into AI agents. Instead of writing every tool yourself, you **connect** an existing server (filesystem, GitHub, Slack, a database ...) and Claude gets all of its tools.

In the *custom tools* notebook you built an **in-process** server. This notebook is about the other kinds, and how to tell when a connection is working.

**What you will learn (and see running):**
1. The three transports: **stdio**, **HTTP/SSE**, **in-process SDK**
2. Writing a tiny **stdio MCP server from scratch** (so MCP stops being magic)
3. Connecting it with `mcp_servers` and calling its tools
4. **Tool naming** (`mcp__<server>__<tool>`) and why MCP tools need `allowed_tools`
5. Loading servers from a **`.mcp.json`** file
6. **Checking connection status**: `connected`, `failed`, `needs-auth`, `pending`
7. Authentication (env vars, headers, OAuth) and timeouts/limits

## The picture

```
 your Python app ── query(options=ClaudeAgentOptions(mcp_servers={...})) ──▶ Claude Code process
                                                                               │
              ┌────────────────────────────────┬──────────────────────────────┤
              ▼                                ▼                              ▼
   stdio server (a command)        HTTP / SSE server (a URL)      SDK server (inside your app)
   {"command": "npx", "args": ...}  {"type": "http", "url": ...}   create_sdk_mcp_server(...)
   local subprocess, JSON over      cloud APIs, remote services    your own @tool functions
   stdin / stdout
```

**How to pick the transport:** the server's docs give you a **command to run** → stdio. They give you a **URL** → HTTP or SSE. You are writing the tools yourself in Python → SDK server.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("mcp-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Imports and a runner that shows the connection

`run()` prints the **MCP server statuses** from the `init` message (the first message of every query), every `mcp__` tool call, and the answer.

In [ ]:
import asyncio, json, os, sys, subprocess
from pathlib import Path

from claude_agent_sdk import (
    query, ClaudeSDKClient, ClaudeAgentOptions,
    AssistantMessage, UserMessage, ResultMessage, SystemMessage,
    ToolUseBlock, ToolResultBlock,
)

async def run(prompt, show_init=True, **opts):
    opts.setdefault("cwd", str(DEMO_DIR))
    opts.setdefault("max_turns", 8)
    final = ""
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, SystemMessage) and m.subtype == "init" and show_init:
                for s in m.data.get("mcp_servers", []):
                    print(f"  [server] {s.get('name')}: {s.get('status')}")
                tools = [t for t in m.data.get("tools", []) if t.startswith("mcp__")]
                print(f"  [mcp tools visible at init] {tools}")
            elif isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        print(f"  [call]   {b.name}({json.dumps(b.input)})")
            elif isinstance(m, UserMessage) and isinstance(m.content, list):
                for b in m.content:
                    if isinstance(b, ToolResultBlock):
                        text = b.content if isinstance(b.content, str) else json.dumps(b.content)
                        print(f"  [result]{' (is_error)' if b.is_error else ''} {str(text)[:160]}")
            elif isinstance(m, ResultMessage):
                if m.subtype == "success":
                    final = m.result
                else:
                    print("  [ended with]", m.subtype)
    except Exception as error:
        print("  (query raised:", str(error)[:200], ")")
    print("ANSWER:", final)
    return final

## 2. Build a stdio MCP server from scratch

A stdio server is just **a program that reads JSON-RPC messages (one per line) from stdin and writes replies to stdout**. Claude Code launches it as a subprocess. Real servers use a library, but the protocol is small enough to write by hand, and doing so shows exactly what happens:

| Message from client | Server must answer with |
|---|---|
| `initialize` | its `protocolVersion`, `capabilities` (`{"tools": {}}`) and `serverInfo` |
| `notifications/initialized` | nothing (it is a notification) |
| `tools/list` | the list of tools, each with `name`, `description`, `inputSchema` |
| `tools/call` | a result with `content` blocks (and optional `isError`) |

Our server is a **notes keeper**: `add_note`, `list_notes`, `search_notes`, stored in a JSON file whose path arrives through an **environment variable** (that is how credentials and config are normally passed to stdio servers).

In [ ]:
SERVER_CODE = r"""
import json, os, sys

STORE = os.environ.get("NOTES_FILE", "notes.json")

def load():
    try:
        return json.load(open(STORE))
    except Exception:
        return []

def save(notes):
    json.dump(notes, open(STORE, "w"))

TOOLS = [
    {"name": "add_note", "description": "Save a short note. Use when the user wants something remembered.",
     "inputSchema": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
    {"name": "list_notes", "description": "List all saved notes with their numbers.",
     "inputSchema": {"type": "object", "properties": {}}},
    {"name": "search_notes", "description": "Find notes containing a keyword (case-insensitive).",
     "inputSchema": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]}},
]

def call(name, args):
    notes = load()
    if name == "add_note":
        notes.append(args["text"]); save(notes)
        return f"Saved note #{len(notes)}"
    if name == "list_notes":
        return "\n".join(f"{i+1}. {n}" for i, n in enumerate(notes)) or "(no notes yet)"
    if name == "search_notes":
        hits = [f"{i+1}. {n}" for i, n in enumerate(notes) if args["keyword"].lower() in n.lower()]
        return "\n".join(hits) or "(no matches)"
    raise ValueError("unknown tool " + name)

def reply(msg_id, result=None, error=None):
    msg = {"jsonrpc": "2.0", "id": msg_id}
    if error is not None:
        msg["error"] = error
    else:
        msg["result"] = result
    sys.stdout.write(json.dumps(msg) + "\n")
    sys.stdout.flush()          # without flush the client waits forever

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    req = json.loads(line)
    method, msg_id = req.get("method"), req.get("id")
    if method == "initialize":
        reply(msg_id, {"protocolVersion": req["params"].get("protocolVersion", "2025-06-18"),
                       "capabilities": {"tools": {}},
                       "serverInfo": {"name": "notes", "version": "1.0.0"}})
    elif method == "tools/list":
        reply(msg_id, {"tools": TOOLS})
    elif method == "tools/call":
        p = req["params"]
        try:
            text = call(p["name"], p.get("arguments") or {})
            reply(msg_id, {"content": [{"type": "text", "text": text}]})
        except Exception as e:
            reply(msg_id, {"content": [{"type": "text", "text": "Error: " + str(e)}], "isError": True})
    elif msg_id is not None:                       # unknown request -> JSON-RPC error
        reply(msg_id, error={"code": -32601, "message": "Method not found: " + str(method)})
    # notifications (no id), e.g. notifications/initialized, need no reply
"""

SERVER_PATH = DEMO_DIR / "notes_server.py"
SERVER_PATH.write_text(SERVER_CODE)
NOTES_FILE = DEMO_DIR / "notes.json"
NOTES_FILE.unlink(missing_ok=True)
print("Wrote", SERVER_PATH)

### Talk to it by hand (no AI, no API cost)

This is exactly what Claude Code does when it connects: start the process, say hello, ask for the tool list, call a tool.

In [ ]:
# OFFLINE
proc = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
    text=True, env={**os.environ, "NOTES_FILE": str(NOTES_FILE)},
)

def rpc(payload):
    proc.stdin.write(json.dumps(payload) + "\n"); proc.stdin.flush()
    return json.loads(proc.stdout.readline()) if "id" in payload else None

print(rpc({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18"}})["result"]["serverInfo"])
rpc({"jsonrpc": "2.0", "method": "notifications/initialized"})            # notification: no reply
print([t["name"] for t in rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})["result"]["tools"]])
print(rpc({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
           "params": {"name": "add_note", "arguments": {"text": "test note"}}})["result"])
print(rpc({"jsonrpc": "2.0", "id": 4, "method": "tools/call",
           "params": {"name": "list_notes", "arguments": {}}})["result"]["content"][0]["text"])
proc.stdin.close(); proc.wait(timeout=5)
NOTES_FILE.unlink(missing_ok=True)       # reset so the AI demos start clean

## 3. Connect it to Claude

A stdio server config is `command` + `args` (+ optional `env`). We use `sys.executable` so it runs with **this notebook's Python**.

The key you choose (`"notes"`) becomes part of every tool name: `mcp__notes__add_note`, and so on. The wildcard `mcp__notes__*` approves them all.

In [ ]:
notes_config = {
    "command": sys.executable,
    "args": [str(SERVER_PATH)],
    "env": {"NOTES_FILE": str(NOTES_FILE)},          # config / secrets go here
}

await run(
    "Save two notes: 'buy oat milk' and 'call the dentist on Friday'. Then list all my notes.",
    mcp_servers={"notes": notes_config},
    allowed_tools=["mcp__notes__*"],
)
print()
print("The server really wrote a file:", NOTES_FILE.read_text())

Read the output top to bottom: the `[server] notes: connected` line comes from the **init** message, then Claude's tool calls, then the answer. The JSON file on disk proves the tools ran in **your** server process, not inside Claude.

A second query is a fresh agent (no memory of the first), but the *file* persists, so it can find the earlier notes:

In [ ]:
await run(
    "Search my notes for 'dentist' and tell me what it says.",
    mcp_servers={"notes": notes_config},
    allowed_tools=["mcp__notes__search_notes"],       # approve ONE tool only
)

## 4. MCP tools need permission

> **Without permission, Claude can *see* that MCP tools exist but cannot *call* them.**

Same request, but we forget `allowed_tools`:

In [ ]:
await run(
    "Use the notes tools to list my notes. If you are not able to call them, say so plainly.",
    mcp_servers={"notes": notes_config},
    # no allowed_tools on purpose
)

In a script there is nobody to click "Approve", so the call is refused and Claude tells you. Your three options:

| Approach | Verdict |
|---|---|
| `allowed_tools=["mcp__notes__*"]` | **Preferred.** Grants exactly one server. |
| `permission_mode="acceptEdits"` | Does **not** cover MCP tools (only file edits). |
| `permission_mode="bypassPermissions"` | Works, but switches off most safety prompts. Broader than needed. |

### The naming rule

`mcp__<server key>__<tool name>`. Characters like `.` or spaces in the server key become `_`:

In [ ]:
# OFFLINE
import re
def mcp_tool_name(server_key, tool):
    return f"mcp__{re.sub(r'[^A-Za-z0-9_-]', '_', server_key)}__{tool}"

for key in ["github", "my.server", "claude-code-docs", "my server"]:
    print(f"{key!r:22} -> {mcp_tool_name(key, 'list_issues')}")

## 5. Loading servers from a `.mcp.json` file

Instead of code, put the config in a **`.mcp.json`** at the project root. It is read when the **`project` setting source** is enabled (the SDK default for `query()` is user+project+local, but if you set `setting_sources` yourself, include `"project"`).

`${VAR}` inside the file expands from environment variables, so secrets stay out of the file.

In [ ]:
mcp_json = {
    "mcpServers": {
        "notes": {
            "command": sys.executable,
            "args": [str(SERVER_PATH)],
            "env": {"NOTES_FILE": "${NOTES_FILE}"},     # expanded at runtime
        }
    }
}
(DEMO_DIR / ".mcp.json").write_text(json.dumps(mcp_json, indent=2))
print((DEMO_DIR / ".mcp.json").read_text())

In [ ]:
await run(
    "List my notes using the notes tools.",
    setting_sources=["project"],                       # <- makes .mcp.json load
    allowed_tools=["mcp__notes__*"],
    env={"NOTES_FILE": str(NOTES_FILE)},
)

A project-level `.mcp.json` may ask for approval the first time, in interactive Claude Code. In the SDK, `allowed_tools` is what lets the calls through. Config files are great for sharing server setups with a team via git; code is better when the config is computed.

## 6. Checking whether a server actually connected

Servers can fail to start (missing program, wrong credentials, network). **That does not raise an exception**. The query simply runs without that server's tools, and Claude may quietly fall back to other tools. So you must **check**.

The `init` message lists each server with a status:

| Status | Meaning |
|---|---|
| `connected` | Working |
| `pending` | Not a failure by itself: still connecting, or tools served from a cache and connected on first use |
| `failed` | Could not connect: server unusable |
| `needs-auth` | Needs credentials (see authentication below) |
| `disabled` | Turned off |

A server whose program does not exist:

In [ ]:
broken_config = {"command": "this-program-does-not-exist-12345", "args": []}

await run(
    "Use the broken server's tools to list notes. If no such tools exist, say so.",
    mcp_servers={"broken": broken_config, "notes": notes_config},
    allowed_tools=["mcp__broken__*", "mcp__notes__*"],
)

You should see `broken: failed` next to `notes: connected`, and the working server was unaffected. In real code, turn this into a guard:

```python
if isinstance(message, SystemMessage) and message.subtype == "init":
    down = [s for s in message.data.get("mcp_servers", []) if s.get("status") in ("failed", "needs-auth")]
    if down:
        print("Unavailable MCP servers:", down)
```

You can also ask a **live client** for current status at any time (status can change mid-session: a remote server may drop to `pending` while reconnecting, and after five failed attempts report `failed`):

In [ ]:
async with ClaudeSDKClient(options=ClaudeAgentOptions(
        cwd=str(DEMO_DIR), mcp_servers={"notes": notes_config}, allowed_tools=["mcp__notes__*"])) as client:
    await client.query("Reply with the single word: ready")
    async for m in client.receive_response():
        pass
    status = await client.get_mcp_status()
    for s in status.get("mcpServers", []):
        print(s.get("name"), "->", s.get("status"), "| tools:", [t.get("name") for t in s.get("tools", [])])
    # await client.reconnect_mcp_server("notes")   # manual retry for a failed server

## 7. Remote servers (HTTP / SSE) and authentication

These cells show **configuration shapes**. They are not run here because they need a network, an account, or a token. Replace the placeholders and run them yourself.

**A public HTTP server (no auth)**, the Claude Code docs server:

In [ ]:
RUN_REMOTE = False       # set True to try the public docs server (needs internet)

docs_config = {"type": "http", "url": "https://code.claude.com/docs/mcp"}

if RUN_REMOTE:
    await run(
        "Use the docs MCP server to explain in two sentences what hooks are in Claude Code.",
        mcp_servers={"claude-code-docs": docs_config},
        allowed_tools=["mcp__claude-code-docs__*"],
    )
else:
    print("Skipped (RUN_REMOTE = False). Config shape:", docs_config)

**Passing credentials: three patterns**

```python
# 1) stdio server: secrets via env (never hard-code them)
{"command": "npx", "args": ["-y", "@your-org/api-mcp-server"],
 "env": {"API_KEY": os.environ["API_KEY"]}}

# 2) HTTP / SSE server: secrets via headers
{"type": "http", "url": "https://api.example.com/mcp",
 "headers": {"Authorization": f"Bearer {os.environ['API_TOKEN']}"}}

# 3) SSE instead of streamable HTTP: change the type only
{"type": "sse", "url": "https://api.example.com/mcp/sse", "headers": {...}}
```

**OAuth:** the SDK never opens a browser. If a server needs OAuth and no token is stored, the run continues *without* that server and its status is `needs-auth`. Do the OAuth flow in **your own app**, then pass the resulting access token in `headers` as above.

**A real-world example (GitHub):**

```python
mcp_servers={"github": {"type": "http", "url": "https://api.githubcopilot.com/mcp/",
                        "headers": {"Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}"}}},
allowed_tools=["mcp__github__list_issues"]      # allow only the one read-only tool you need
```

**Databases:** a database server runs whatever SQL Claude writes. Restrict it at the *server* (for example a read-only flag in its config), not just in the prompt.

## 8. Timing, timeouts and size limits

| Topic | What to know |
|---|---|
| **First-turn wait** | stdio servers, and HTTP/SSE servers with no cached tool list, **delay the first turn** until they connect, up to `MCP_TIMEOUT` (default 30 s) |
| **Cached remote servers** | If Claude Code cached the tool list from a previous connection, the first turn does not wait; it connects on first use |
| **Slow starters** | Raise `MCP_TIMEOUT` (milliseconds) via the `env` option; or `CLAUDE_CODE_MCP_STARTUP_WAIT_MS` to set the first-turn wait yourself (`0` = don't wait) |
| **Tool call duration** | `MCP_TOOL_TIMEOUT` |
| **Big outputs** | A successful result over ~25,000 tokens is saved to a file and Claude gets an error naming the path; raise with `MAX_MCP_OUTPUT_TOKENS`. Text over 50,000 characters is also saved to a file |
| **Many tools** | Tool search is on by default and defers tool definitions. See the *tool search* notebook |

Setting environment variables for the Claude Code process:

```python
ClaudeAgentOptions(env={"MCP_TIMEOUT": "60000", "MAX_MCP_OUTPUT_TOKENS": "50000"}, ...)
```

## 9. Troubleshooting checklist

| Symptom | Likely cause and fix |
|---|---|
| Server shows `failed` | Program not installed or not on PATH (`npx` needs Node.js), missing env var, bad connection string, URL unreachable, or startup slower than `MCP_TIMEOUT` |
| `pending` at init | Often normal; check again later with `get_mcp_status()` |
| Claude sees tools but never uses them | Missing `allowed_tools` entry (use `mcp__<key>__*`) |
| Tool name not matching | Server key contains `.` or space, which became `_` |
| `.mcp.json` ignored | `setting_sources` set without `"project"` |
| Answer looks like Claude improvised | Server failed silently and Claude fell back to built-ins: check the init statuses |

## 10. Summary

| I want to ... | Do this |
|---|---|
| Run a local server program | `{"command": ..., "args": [...], "env": {...}}` |
| Connect a hosted server | `{"type": "http", "url": ..., "headers": {...}}` |
| Write tools in Python | `create_sdk_mcp_server` (see the *custom tools* notebook) |
| Let Claude call the tools | `allowed_tools=["mcp__<key>__*"]` |
| Share config in a repo | `.mcp.json` + `setting_sources=["project"]` |
| Know it worked | Check `init` statuses or `get_mcp_status()` |